In [6]:
# SemEval-2027 Task 3, Subtask 2: submission pipeline
#   MODE = "dry_run": runs the full pipeline on Swedish dev and verifies it with the official scorer.
#   MODE = "test":    scores the test data in TEST_DIR and writes the submission file.
MODE = "dry_run"
TEST_DIR = "work/data/test"          # used only in test mode (same folder layout as work/data/dev)

import sys, gc, json, time, hashlib, subprocess, importlib, importlib.util
from collections import Counter
from importlib.metadata import version
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in (PROJECT / "src", PROJECT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bcubed"], check=True)
importlib.invalidate_caches()
import numpy as np, torch
from peft import PeftModel
from peft.import_utils import is_torchao_available
from project_utils import save_json
from semeval_task3.data import load_dataset
from semeval_task3.evaluation import write_submission, read_submission, official_score

assert MODE in ("dry_run", "test"), 'MODE must be "dry_run" or "test".'
DRY = MODE == "dry_run"
SYSTEM = PROJECT / "outputs/frozen/main_system_ensemble5_gemma4_31b_lora.json"
INPUT_DIR = PROJECT / ("work/data/dev" if DRY else TEST_DIR)
OUT = PROJECT / "outputs/submissions" / ("dry_run_SV_dev" if DRY else "test_phase2")
SUBMISSION = OUT / "submission_gemma4_ensemble5.zip"
R1_RUN = PROJECT / "outputs/runs/step09_lora_full_pilot/20261105T232211Z_6b04a51b"
ENS_ROOT = PROJECT / "outputs/runs/step15_seed_ensemble"
STEP19 = PROJECT / "outputs/runs/step19_five_seeds"
R1_DATA = PROJECT / "outputs/runs/step07_pilot_dataset/20261105T223146Z_1dd5a63b"
ENRU_EVAL = PROJECT / "outputs/runs/step07_pilot_evaluation/20261105T223700Z_5259f45a"
XL_EVAL = PROJECT / "outputs/runs/step11_xlwsd_evaluation_sets/20261106T032028Z_aa4e2120"
SCORER_DIR, SCORER_COMMIT = Path("/content/official-scorer-latest"), "0e69238"
LANG_NAME = dict(SV="Swedish", EN="English", IT="Italian", ES="Spanish", NL="Dutch", RU="Russian")
DEVICE, YES_ID, NO_ID = "cuda", 10784, 3771
STEPS = 7

def header(n, title):
    print(f"\n[{n}/{STEPS}] {title}\n" + "-" * 78, flush=True)

def line(label, value):
    print(f"  {label:<40}{value}", flush=True)

def check(label, ok, detail="", hint=""):
    print(f"  {label:<40}{'PASS' if ok else 'FAIL'}{'   ' + detail if detail else ''}", flush=True)
    if not ok:
        raise RuntimeError(f"Check failed: {label}. {hint or detail}".strip())

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def read_rows(path):
    with Path(path).open(encoding="utf-8") as handle:
        return [json.loads(x) for x in handle if x.strip()]

def system_prompt(lang):
    return f"You are an expert lexicographer of historical {LANG_NAME[lang]}."

def messages_for(u):
    lang = u.key[0]
    return [dict(role="system", content=system_prompt(lang)),
            dict(role="user", content=(f"Target word (lemma): {u.word}\n"
                                       f"Sense definition ({LANG_NAME[lang]}): {u.definition}\n"
                                       f"Sentence from {u.year}: {u.marked_text()}\n\n"
                                       "Does the word marked with <t> and </t> in the sentence express the sense given by the definition? "
                                       "Answer only Yes or No."))]

def identity(rows):
    return hashlib.sha256(json.dumps([(str(r["id"]), r["messages"]) for r in rows],
                                     ensure_ascii=False, sort_keys=True).encode()).hexdigest()

print("=" * 78)
print("SemEval-2027 Task 3, Subtask 2: submission pipeline")
print(f"Mode: {'dry run on Swedish dev (verification)' if DRY else 'TEST: writing the submission file'}")
print(f"Started: {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC")
print("=" * 78)

# ---------------------------------------------------------------------------
header(1, "Frozen system and adapter fingerprints")
is_torchao_available.cache_clear()
check("peft environment (no torchao)", is_torchao_available() is False, hint="Restart the session and run again.")
cfg = json.loads(SYSTEM.read_text())
THRESHOLD = float(cfg["final_threshold"])
adapters = cfg["adapters"]
seeds = [a["seed"] for a in adapters]
line("System", f"average of {len(adapters)} LoRA adapters (seeds {seeds[0]}-{seeds[-1]})")
line("Base model", f"{cfg['model_id']} @ {cfg['revision'][:12]}")
line("Decision rule", f"predict 1 if average score >= {THRESHOLD:+.2f}")
check("Threshold is 1.00", THRESHOLD == 1.0)
for a in adapters:
    check(f"Adapter {a['seed']} fingerprint", digest(Path(a["adapter_dir"]) / "adapter_model.safetensors") == a["sha256"],
          a["sha256"][:16])

# ---------------------------------------------------------------------------
header(2, "Input data and prompts")
ds = load_dataset(INPUT_DIR, with_gold=DRY)
usages = list(ds.usages)
keys = [u.key for u in usages]
rows = [dict(id="|".join(u.key), messages=messages_for(u)) for u in usages]
ids = [r["id"] for r in rows]
check("Unique usage ids", len(set(ids)) == len(ids))
langs = Counter(u.key[0] for u in usages)
check("Languages supported", set(langs) <= set(LANG_NAME), ", ".join(sorted(langs)))
line("Input folder", str(INPUT_DIR.relative_to(PROJECT)))
for lang in sorted(langs):
    n_words = len({u.word for u in usages if u.key[0] == lang})
    line(f"  {lang} ({LANG_NAME[lang]})", f"{langs[lang]:,} usages, {n_words} words")
line("Total", f"{len(rows):,} usages")
if DRY:
    stored = json.loads((STEP19 / "seed_2030/eval_scores/SV_dev.json").read_text())["input_sha256"]
    check("Prompts identical to evaluation runs", identity(rows) == stored)
seen = Counter()
for path in [R1_DATA / "pilot_train.jsonl", ENRU_EVAL / "EN_wic_dev.jsonl", ENRU_EVAL / "RU_axolotl_test_derived.jsonl",
             XL_EVAL / "IT_xlwsd_test.jsonl", XL_EVAL / "ES_xlwsd_test.jsonl", XL_EVAL / "NL_xlwsd_test.jsonl"]:
    for r in read_rows(path):
        seen[r["messages"][0]["content"]] += 1
print("  Prompt template vs. prompts used in training and evaluation:")
for lang in LANG_NAME:
    n = seen.get(system_prompt(lang), 0)
    status = "matches" if n else ("matches (Swedish dev)" if lang == "SV" else "WARN: no stored prompt with this wording")
    print(f"    {lang}  {system_prompt(lang):<58}{status}{f' ({n:,} rows)' if n else ''}")

# ---------------------------------------------------------------------------
header(3, "Score cache")
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "scores").mkdir(exist_ok=True)
rows_id = identity(rows)
def cached(seed):
    p = OUT / "scores" / f"seed_{seed}.json"
    if not p.exists():
        return False
    s = json.loads(p.read_text())
    return s["input_sha256"] == rows_id and len(s["scores"]) == len(rows)
done = {s: cached(s) for s in seeds}
for s in seeds:
    line(f"Seed {s}", "complete (cached)" if done[s] else "to score")
line("Output folder", str(OUT.relative_to(PROJECT)))

# ---------------------------------------------------------------------------
header(4, "Model")
if all(done.values()):
    line("Gemma 4 31B", "not needed (all scores cached)")
else:
    if "gemma_model" not in globals():
        from transformers import AutoProcessor, AutoModelForMultimodalLM
        props = torch.cuda.get_device_properties(0)
        line("GPU", f"{props.name}, {props.total_memory / 2**30:.0f} GiB")
        check("GPU memory above 70 GiB", props.total_memory / 2**30 > 70, hint="Use the G4 or an 80 GB A100.")
        t0 = time.perf_counter()
        gemma_processor = AutoProcessor.from_pretrained(cfg["model_id"], revision=cfg["revision"])
        gemma_model = AutoModelForMultimodalLM.from_pretrained(cfg["model_id"], revision=cfg["revision"],
                                                               dtype=torch.bfloat16, device_map={"": 0}, attn_implementation="sdpa")
        line("Gemma 4 31B", f"loaded in {(time.perf_counter() - t0) / 60:.1f} min, transformers {version('transformers')}")
    else:
        line("Gemma 4 31B", "already loaded in this session")
    gemma_processor.tokenizer.padding_side = "left"
    check("Yes/No token ids", gemma_processor.tokenizer.encode("Yes", add_special_tokens=False) == [YES_ID]
          and gemma_processor.tokenizer.encode("No", add_special_tokens=False) == [NO_ID])
    check("Base model has no adapter attached", not any("lora_" in n for n, _ in gemma_model.named_parameters()))

def encode(batch):
    inputs = gemma_processor.apply_chat_template(
        [r["messages"] for r in batch], tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
        processor_kwargs={"padding": True, "truncation": False}).to(DEVICE)
    assert inputs["attention_mask"][:, -1].all(), "Padding is not on the left."
    return inputs

def score_rows(model, destination, tag):
    saved = json.loads(destination.read_text()) if destination.exists() else dict(input_sha256=rows_id, model_tag=tag, scores={})
    assert saved["input_sha256"] == rows_id and saved["model_tag"] == tag, f"{destination.name} belongs to other inputs."
    pending = [r for r in rows if r["id"] not in saved["scores"]]
    model.eval()
    try:
        with torch.inference_mode(), torch.autocast(DEVICE, dtype=torch.bfloat16):
            for b, i in enumerate(range(0, len(pending), 2), 1):
                batch = pending[i:i + 2]
                logits = model(**encode(batch), use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
                assert torch.isfinite(logits).all(), "Non-finite logits."
                for r, m in zip(batch, (logits[:, YES_ID] - logits[:, NO_ID]).tolist()):
                    saved["scores"][r["id"]] = dict(margin=float(m))
                del logits
                if b % 100 == 0:
                    save_json(destination, saved)
    finally:
        save_json(destination, saved)

# ---------------------------------------------------------------------------
header(5, "Scoring (one pass per adapter)")
for a in adapters:
    seed, dest = a["seed"], OUT / "scores" / f"seed_{a['seed']}.json"
    if done[seed]:
        line(f"Seed {seed}", "cached")
        continue
    t0 = time.perf_counter()
    model = PeftModel.from_pretrained(gemma_model, a["adapter_dir"], is_trainable=False)
    score_rows(model, dest, f"seed_{seed}")
    gemma_model = model.unload()
    del model
    gc.collect(); torch.cuda.empty_cache()
    check(f"Seed {seed} scored", cached(seed), f"{len(rows):,} usages in {(time.perf_counter() - t0) / 60:.1f} min")
    check(f"Seed {seed} adapter removed", not any("lora_" in n for n, _ in gemma_model.named_parameters()))
S = np.array([[json.loads((OUT / "scores" / f"seed_{s}.json").read_text())["scores"][i]["margin"] for i in ids] for s in seeds])
check("All scores finite", bool(np.isfinite(S).all()))

# ---------------------------------------------------------------------------
header(6, "Submission file")
expected_keys = set(keys)
avg = S.mean(axis=0)
pred = {k: int(v >= THRESHOLD) for k, v in zip(keys, avg)}
for old in OUT.glob("submission_*.zip"):
    if old != SUBMISSION:
        old.unlink()
write_submission(SUBMISSION, pred)
check("File reads back identically", read_submission(SUBMISSION, expected_keys) == pred,
      f"{SUBMISSION.name}, {SUBMISSION.stat().st_size / 1024:.0f} KB")
print(f"\n  {'Language':<12}{'usages':>10}{'predicted Yes':>16}")
for lang in sorted(langs):
    ks = [k for k in keys if k[0] == lang]
    print(f"  {lang:<12}{len(ks):>10,}{np.mean([pred[k] for k in ks]):>16.1%}")
save_json(OUT / "scores/average_scores.json", {i: float(v) for i, v in zip(ids, avg)})

# ---------------------------------------------------------------------------
header(7, "Verification against the official scorer" if DRY else "Test-mode summary")
if DRY:
    def stored(seed):
        if seed == 2027:
            d = json.loads((R1_RUN / "SV_scores.json").read_text())["scores"]
        else:
            root = ENS_ROOT if seed in (2028, 2029) else STEP19
            d = json.loads((root / f"seed_{seed}/eval_scores/SV_dev.json").read_text())["scores"]
        return np.array([d[i]["margin"] for i in ids])
    print(f"  {'Agreement with earlier scoring':<28}{'mean |diff|':>12}{'max |diff|':>12}{'correlation':>13}{'same answer':>13}")
    old_all = []
    for n, s in enumerate(seeds):
        old = stored(s)
        old_all.append(old)
        diff = np.abs(S[n] - old)
        corr = float(np.corrcoef(S[n], old)[0, 1])
        same = float(np.mean((S[n] >= THRESHOLD) == (old >= THRESHOLD)))
        print(f"    seed {s:<22}{diff.mean():>12.4f}{diff.max():>12.4f}{corr:>13.5f}{same:>13.2%}")
        check(f"  seed {s} reproduces its scores", corr > 0.999, f"correlation {corr:.5f}",
              hint="Scores differ beyond rounding noise.")
    old_avg = np.mean(old_all, axis=0)
    same_sys = float(np.mean((avg >= THRESHOLD) == (old_avg >= THRESHOLD)))
    check("Submitted answers match earlier run", same_sys >= 0.99,
          f"{same_sys:.2%} identical ({int(round((1 - same_sys) * len(ids)))} of {len(ids):,} flipped)")
    print("  Small differences come from bf16 rounding on different GPUs (scores move in steps of 0.125).")
    if not SCORER_DIR.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/ChangeIsKey/semeval2027-task3.git", str(SCORER_DIR)], check=True)
    subprocess.run(["git", "-C", str(SCORER_DIR), "checkout", "-q", SCORER_COMMIT], check=True)
    sys.path.insert(0, str(SCORER_DIR))
    spec = importlib.util.spec_from_file_location("official_eval_latest", SCORER_DIR / "evaluation.py")
    official = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(official)
    gold = ds.require_gold()
    report = official_score(SUBMISSION, INPUT_DIR, SCORER_DIR)
    check("Official scorer accepts the file", not report.get("errors"))
    restored = read_submission(SUBMISSION, expected_keys)
    score = official.compute_f1_score({f"{k[1]};{k[2]}": int(v) for k, v in gold.items()},
                                      {f"{k[1]};{k[2]}": int(restored[k]) for k in gold}, "check")
    frozen = cfg["swedish_in_sample_macro_f1"]
    check("Macro-F1 matches the frozen value", abs(score - frozen) < 0.005, f"{score:.4f} vs {frozen:.4f}")
    save_json(OUT / "dry_run_report.json", dict(mode=MODE, utc=datetime.now(timezone.utc).isoformat(),
                                                  macro_f1=score, file=str(SUBMISSION)))
else:
    line("Submission file", str(SUBMISSION))
    save_json(OUT / "test_report.json", dict(mode=MODE, utc=datetime.now(timezone.utc).isoformat(), usages=len(rows),
                                             languages=dict(langs), file=str(SUBMISSION)))

print("\n" + "=" * 78)
print("Summary")
print("=" * 78)
line("Mode", "dry run (Swedish dev)" if DRY else "test")
line("Usages scored", f"{len(rows):,} across {len(langs)} language(s)")
line("Submission file", SUBMISSION.name)
if DRY:
    line("Swedish dev macro-F1", f"{score:.4f}")
    line("Status", "pipeline verified; ready for the test phase")
else:
    line("Status", "file written; ready to upload to Codabench")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SemEval-2027 Task 3, Subtask 2: submission pipeline
Mode: dry run on Swedish dev (verification)
Started: 2026-11-17 02:00 UTC

[1/7] Frozen system and adapter fingerprints
------------------------------------------------------------------------------
  peft environment (no torchao)           PASS
  System                                  average of 5 LoRA adapters (seeds 2027-2031)
  Base model                              google/gemma-4-31B-it @ 842da3794eaa
  Decision rule                           predict 1 if average score >= +1.00
  Threshold is 1.00                       PASS
  Adapter 2027 fingerprint                PASS   6f3d236ac50e07cc
  Adapter 2028 fingerprint                PASS   9b6d485af7177301
  Adapter 2029 fingerprint                PASS   bd5c266eff0d57f0
  Adapter 2030 fingerprint                PASS   a6e593baf7a18852
  Adapter 2031 fin